# PyTorch Dataset i prvi prolaz kroz model

Od jedne poruke pravimo tenzore, zatim batch i izlaz `SMSClassifier` modela. Model još nije treniran: prikazani skorovi nisu pouzdana predviđanja.

## Mali primer sa dve poruke

Oznaka klase je druga vrsta broja od ID-ja tokena: `ham = 0`, `spam = 1`. Rečnik tokena gradimo samo iz poruka koje nam služe za učenje.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.sms_dataset import LABEL_TO_ID, SMSDataset
from src.vocabulary import build_vocabulary

toy = pd.DataFrame({"label": ["ham", "spam"], "text": ["See you soon", "Free prize now!"]})
toy_vocabulary = build_vocabulary(toy["text"])
toy_dataset = SMSDataset(toy, toy_vocabulary, max_length=8)
example = toy_dataset[1]
print("Mapiranje oznaka:", LABEL_TO_ID)
pd.DataFrame({
    "polje": list(example),
    "tip": [str(value.dtype) for value in example.values()],
    "vrednost": [value.tolist() for value in example.values()],
})

Mapiranje oznaka: {'ham': 0, 'spam': 1}


,polje,tip,vrednost
0,input_ids,torch.int64,"[3, 5, 4, 2, 0, 0, 0, 0]"
1,padding_mask,torch.bool,"[False, False, False, False, True, True, True,..."
2,label,torch.int64,1


U jednom primeru `input_ids` i `padding_mask` imaju po 8 vrednosti. `label` je jedan broj. Maska je tipa `torch.bool`; ID-jevi i oznaka su celobrojni tenzori.

In [2]:
toy_loader = DataLoader(toy_dataset, batch_size=2, shuffle=False)
toy_batch = next(iter(toy_loader))
pd.DataFrame({
    "polje": list(toy_batch),
    "oblik": [str(tuple(value.shape)) for value in toy_batch.values()],
    "tip": [str(value.dtype) for value in toy_batch.values()],
})

,polje,oblik,tip
0,input_ids,"(2, 8)",torch.int64
1,padding_mask,"(2, 8)",torch.bool
2,label,"(2,)",torch.int64


Prva dimenzija `2` znači dve poruke u batch-u, druga dimenzija `8` znači osam pozicija po poruci. Oblik oznaka je `(2,)`: po jedna oznaka za svaku poruku. `shuffle=False` ovde olakšava pregled; tokom stvarnog treniranja redosled ćemo mešati.

## Prvi batch iz trening skupa

Ovde gradimo rečnik samo iz `sms_train.csv`, pa pravimo `SMSDataset` nad istim trening porukama. Izdvojeni test se ne čita.

In [3]:
train = pd.read_csv(project_root / "data" / "processed" / "sms_train.csv")
vocabulary = build_vocabulary(train["text"])
train_dataset = SMSDataset(train, vocabulary, max_length=128)
train_batch = next(iter(DataLoader(train_dataset, batch_size=4, shuffle=False)))
print("Broj poruka u trening Dataset-u:", len(train_dataset))
print("Oblici:", {name: tuple(value.shape) for name, value in train_batch.items()})
print("Oznake prvog batch-a:", train_batch["label"].tolist())

Broj poruka u trening Dataset-u: 4127
Oblici: {'input_ids': (4, 128), 'padding_mask': (4, 128), 'label': (4,)}
Oznake prvog batch-a: [0, 0, 1, 0]


## Prvi prolaz kroz Transformer

Batch iz malog primera ulazi u embedding, Transformer encoder i klasifikator. Tačne oznake klasa ne ulaze u modelov `forward`; biće potrebne za računanje greške pri treniranju.

In [4]:
from src.model import SMSClassifier

print("ID-jevi:", toy_batch["input_ids"].tolist())
print("Maska:", toy_batch["padding_mask"].tolist())

ID-jevi: [[6, 8, 7, 0, 0, 0, 0, 0], [3, 5, 4, 2, 0, 0, 0, 0]]
Maska: [[False, False, False, True, True, True, True, True], [False, False, False, False, True, True, True, True]]


## Slojevi i oblici

Embedding za svaki token ima 64 broja. Pozicioni embedding iste dimenzije dodaje informaciju o redosledu. Dva Transformer bloka zatim računaju kontekstualne vektore. Prosek uzima samo stvarne tokene, a poslednji linearni sloj vraća **dva sirova skora** po poruci.

In [5]:
torch.manual_seed(42)
model = SMSClassifier(vocab_size=len(toy_vocabulary), max_length=8)
model.eval()
with torch.no_grad():
    token_vectors = model.token_embedding(toy_batch["input_ids"])
    positions = torch.arange(toy_batch["input_ids"].shape[1]).unsqueeze(0)
    position_vectors = model.position_embedding(positions)
    logits = model(toy_batch["input_ids"], toy_batch["padding_mask"])

pd.DataFrame({
    "deo": ["input_ids", "token embedding", "pozicioni embedding", "logits"],
    "oblik": [str(tuple(tensor.shape)) for tensor in (toy_batch["input_ids"], token_vectors, position_vectors, logits)],
})

,deo,oblik
0,input_ids,"(2, 8)"
1,token embedding,"(2, 8, 64)"
2,pozicioni embedding,"(1, 8, 64)"
3,logits,"(2, 2)"


Oblik pozicionog embedding-a je `(1, 8, 64)`: isti skup pozicija koristi se za obe poruke u batch-u. Modelov izlaz je `(2, 2)` — dve poruke, po dva skora. `padding_mask=True` označava mesta dopune koja se ne koriste kao ključevi u attention-u niti ulaze u prosek poruke.

In [6]:
with torch.no_grad():
    proportions = torch.softmax(logits, dim=1)

pd.DataFrame({
    "poruka": toy["text"],
    "ham skor": logits[:, 0].tolist(),
    "spam skor": logits[:, 1].tolist(),
    "ham udeo": proportions[:, 0].tolist(),
    "spam udeo": proportions[:, 1].tolist(),
})

,poruka,ham skor,spam skor,ham udeo,spam udeo
0,See you soon,0.652678,-0.141704,0.688772,0.311228
1,Free prize now!,0.016553,-0.214220,0.557439,0.442561


`softmax` samo pretvara dva skora u brojeve koji sabrani daju 1. Pošto parametri još nisu učeni, ovi brojevi **nisu pouzdane verovatnoće** i ne ocenjujemo model na osnovu njih. Poznate oznake iz batch-a biće potrebne u narednoj celini, kada dodamo funkciju gubitka i treniranje. Izdvojeni test ovde ne koristimo.